PHYS 4730 Fall 2025

Lab 13

In [ ]:
%reset -f

import numpy as np
from numpy import random
import matplotlib.pyplot as plt

import matplotlib
matplotlib.rcParams.update({'font.size': 18})

from IPython.display import display, Math, Latex

In [ ]:
### uniform distribution ###

aa = 1.0
bb = 2.0
nbins = 10
N = 10000
x = random.uniform(aa,bb,size=N)

# plots

plt.figure(figsize=(6,6))
n,b,p = plt.hist(x, bins=nbins, range=[aa,bb])
plt.xlabel("$x$")
plt.ylabel("counts/bin")
plt.grid()
plt.show()

In [ ]:
### Now, weigh events by 1/(N*Delta-x) ###
### This gives a data-derived estimate ###
### for the pdf, or an "empirical pdf" ###

Deltax = (bb-aa)/nbins

plt.figure(figsize=(6,6))

n,b,p = plt.hist(x, bins=nbins, range=[aa,bb], weights=0*x + 1.0/(N*Deltax))
plt.xlabel("$x$")
plt.ylabel("pdf")
plt.grid()

from scipy import stats
dist = stats.uniform(aa,(bb-aa))    # arguments are left edge, width

# compare the 'true' pdf
    
xx = np.arange(aa,bb,0.1*Deltax)
pdf  = dist.pdf(xx)
plt.plot(xx,pdf,color='r')

plt.show()

In [ ]:
### Another way of doing the same thing ###

Deltax = (bb-aa)/nbins

plt.figure(figsize=(6,6))

n,b,p = plt.hist(x, bins=nbins, range=[aa,bb], density=True)  # <--- this call changed 
plt.xlabel("$x$")
plt.ylabel("pdf")
plt.grid()

from scipy import stats
dist = stats.uniform(aa,(bb-aa))    # arguments are left edge, width
 
xx = np.arange(aa,bb,0.1*Deltax)
pdf = dist.pdf(xx)
plt.plot(xx,pdf,color='r')

plt.show()

In [ ]:
### Another way to look at the data; "cumulative distribution" ###

Deltax = (bb-aa)/nbins

plt.figure(figsize=(6,6))

n,b,p = plt.hist(x, bins=nbins, range=[aa,bb], density=True, cumulative=True)  # <--- this call changed 
plt.xlabel("$x$")
plt.ylabel("cdf")
plt.grid()

from scipy import stats
dist = stats.uniform(aa,(bb-aa))    # arguments are left edge, width
 
xx = np.arange(aa,bb,0.1*Deltax)
cdf = dist.cdf(xx)                                                             # <--- changed!!! 
plt.plot(xx,cdf,color='r')

plt.show()

Confidence Levels and the Normal (Gaussian) Distribution

$$
f(x|\mu,\sigma) = \frac{1}{\sqrt{2\pi}\sigma}e^{-(x-\mu)^2/2\sigma^2}
$$


In [ ]:
### plot a basic Gaussian ###

aa   = -10.0
bb   =  10.0
xloc   =  0.0
scale  =  1.0
Deltax = 0.001

plt.figure(figsize=(6,6))

xx = np.arange(aa,bb,Deltax)
dist = stats.norm(xloc,scale)
pdf = dist.pdf(xx)

plt.plot(xx,pdf,color='r')
plt.xlim(aa,bb)
plt.xlabel("$x$")
plt.ylabel("Gaussian PDF")
plt.show()

In [ ]:
### Plot the "quantile" or percent point function (ppf) ###
### of the Gaussian. This is known as the "probit" or   ###
### "probability unit" function, historically.          ###

pprob = np.arange(0.0,1.0,0.001)
quant = stats.norm.ppf(pprob,xloc,scale)

plt.figure(figsize=(6,6))

plt.plot(pprob,quant,color='r')
plt.xlabel("probability")
plt.ylabel("Gaussian PPF")
plt.show()

In [ ]:
### Q: For a Gaussian, what is probability of a measurement falling ###
### within 1 sigma from the mean? (Could do the integral, but let's ###
### introduce some Python tools.)                                   ### 

### define a function that we will "zero" to find integral from PPF ###

def func_p1(prob):
    xloc  =  0.0
    scale =  1.0
    nsig  =  1.0
    quant = stats.norm.ppf(prob,xloc,scale)
    return quant - nsig

In [ ]:
### Use Brent’s method (see e.g. Numerical Recipes ###
### to find the value.                             ###

from scipy import optimize
rootp1 = optimize.brentq(func_p1,0.0,1.0)
print(rootp1)

In [ ]:
### That is the probability of obtaining x < 1 sigma ###
### We also need p(x < -1 sigma)                     ###

def func_m1(prob):
    xloc  =  0.0
    scale =  1.0
    nsig  =  1.0
    quant = stats.norm.ppf(prob,xloc,scale)
    return (quant + 1.0)

rootm1 = optimize.brentq(func_m1,0.0,1.0)
print(rootm1)

In [ ]:
### The difference between these is the probability of ###
### a measurement within one sigma of the mean.        ###

print("Probability of measurement within one sigma: %6.4f" % (rootp1-rootm1))

In [ ]:
### Now a plot ###

plt.figure(figsize=(6,6))

mu, sigma = 0.0, 1.0

x  = np.linspace(-5.0,5.0,1000)
y  = (1/np.sqrt(2*np.pi))*np.exp(-0.5*x*x)
plt.plot(x,y,'c')
plt.fill_between(x, 0, y, color='c')

y1 = np.copy(y) 
y1[(x < -1.0*sigma) | (x > 1.0*sigma)] = 0
labeltext = "%5.2f" % (100.0*(rootp1-rootm1)) + "\u0025"
plt.plot(x,y1, 'b',label=labeltext)
plt.fill_between(x, 0, y1, color='b')

plt.legend(loc='upper right')
plt.xlabel("$x$")
plt.ylabel("Gaussian PDF")
plt.show()